In [2]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/Apprentissage/DS/projet')

for file in sorted(PROJECT_DIR.rglob('*')):
    if file.is_file() and file.suffix.lower() in ('.keras', '.h5'):
        print(file.relative_to(PROJECT_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
E4_VGG16_best.keras
E4_finetuning/E4_FT_B5_LR1e-5_best.keras
M4_VGG16_multiclass_best.keras
M4_VGG16_multiclass_best_compatible.keras
M4_finetuning/M4_FT2_B5_LR1e-6_best.keras
M4_finetuning/M4_FT_B5_LR1e-5_best.keras
M4_finetuning/M4_VGG16_multiclass_best_compatible.keras


In [3]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path('/content/drive/MyDrive/Apprentissage/DS/projet')

# Rechercher les fichiers de split
for f in sorted(PROJECT_DIR.rglob('*.csv')):
    if any(word in f.name.lower() for word in ['train', 'val', 'test']):
        print(f.relative_to(PROJECT_DIR))

E4_interpretabilite_GradCAM/E4_GradCAM_selection_validation.csv
E4_optimisation_seuil_COVID/baseline_validation.csv
E4_optimisation_seuil_COVID/comparaison_seuils_validation.csv
M4_optimisation_seuil_COVID/baseline_validation.csv
M4_optimisation_seuil_COVID/comparaison_seuils_validation.csv
test.csv
train.csv
validation.csv


In [4]:
import pandas as pd

test_df = pd.read_csv(PROJECT_DIR / "test.csv")

print("Dimensions :", test_df.shape)
print("\nColonnes :", test_df.columns.tolist())
print("\nPremières lignes :")
display(test_df.head())

print("\nValeurs manquantes :")
print(test_df.isna().sum())

Dimensions : (4233, 9)

Colonnes : ['class', 'filename', 'width', 'height', 'mode', 'mean_intensity', 'contrast', 'filepath', 'target']

Premières lignes :


,class,filename,width,height,mode,mean_intensity,contrast,filepath,target
0,Lung_Opacity,Lung_Opacity-121.png,299,299,L,150.562824,63.614784,..\covid\COVID-19_Radiography_Dataset\Lung_Opa...,0
1,Normal,Normal-828.png,299,299,L,137.601313,69.508914,..\covid\COVID-19_Radiography_Dataset\Normal\i...,0
2,Lung_Opacity,Lung_Opacity-4959.png,299,299,L,114.089071,48.265802,..\covid\COVID-19_Radiography_Dataset\Lung_Opa...,0
3,Lung_Opacity,Lung_Opacity-405.png,299,299,L,121.044552,55.605214,..\covid\COVID-19_Radiography_Dataset\Lung_Opa...,0
4,Normal,Normal-3816.png,299,299,L,120.774130,66.938557,..\covid\COVID-19_Radiography_Dataset\Normal\i...,0



Valeurs manquantes :
class             0
filename          0
width             0
height            0
mode              0
mean_intensity    0
contrast          0
filepath          0
target            0
dtype: int64


In [5]:
print("Répartition des classes et des targets :")
print(
    test_df.groupby(["class", "target"])
    .size()
    .reset_index(name="nb_images")
    .to_string(index=False)
)

print("\nExemples de chemins :")
print(test_df["filepath"].head(5).to_string(index=False))

print("\nTypes de données :")
print(test_df[["class", "target", "filepath"]].dtypes)

Répartition des classes et des targets :
          class  target  nb_images
          COVID       1        723
   Lung_Opacity       0       1140
         Normal       0       2090
Viral Pneumonia       0        280

Exemples de chemins :
..\covid\COVID-19_Radiography_Dataset\Lung_Opac...
..\covid\COVID-19_Radiography_Dataset\Normal\im...
..\covid\COVID-19_Radiography_Dataset\Lung_Opac...
..\covid\COVID-19_Radiography_Dataset\Lung_Opac...
..\covid\COVID-19_Radiography_Dataset\Normal\im...

Types de données :
class       object
target       int64
filepath    object
dtype: object


In [6]:
from pathlib import Path

DATASET_DIR = Path("/content/COVID-19_Radiography_Dataset")

print("Dataset présent :", DATASET_DIR.exists())

if DATASET_DIR.exists():
    print("\nDossiers disponibles :")
    for p in sorted(DATASET_DIR.iterdir()):
        if p.is_dir():
            print("-", p.name)

print("\nExemple de chemin CSV :")
print(repr(test_df["filepath"].iloc[0]))

print("\nExemple de nom de fichier :")
print(test_df["filename"].iloc[0])

Dataset présent : False

Exemple de chemin CSV :
'..\\covid\\COVID-19_Radiography_Dataset\\Lung_Opacity\\images\\Lung_Opacity-121.png'

Exemple de nom de fichier :
Lung_Opacity-121.png


In [7]:
from pathlib import Path

locations = [
    Path("/content"),
    Path("/content/drive/MyDrive/Apprentissage/DS"),
]

for root in locations:
    print(f"\n--- Recherche dans {root} ---")

    if not root.exists():
        print("Emplacement absent")
        continue

    matches = [
        p for p in root.rglob("COVID-19_Radiography_Dataset")
        if p.is_dir()
    ]

    if matches:
        for p in matches:
            print("Dataset trouvé :", p)
    else:
        print("Dataset non trouvé")


--- Recherche dans /content ---
Dataset trouvé : /content/drive/.Encrypted/MyDrive/Apprentissage/DS/projet/COVID-19_Radiography_Dataset
Dataset trouvé : /content/drive/MyDrive/Apprentissage/DS/projet/COVID-19_Radiography_Dataset

--- Recherche dans /content/drive/MyDrive/Apprentissage/DS ---
Dataset trouvé : /content/drive/MyDrive/Apprentissage/DS/projet/COVID-19_Radiography_Dataset


In [8]:
from pathlib import Path

DATASET_DIR = PROJECT_DIR / "COVID-19_Radiography_Dataset"

# Reconstruire les chemins Linux à partir des classes et noms de fichiers
test_df["image_path"] = test_df.apply(
    lambda row: str(
        DATASET_DIR / row["class"] / "images" / row["filename"]
    ),
    axis=1
)

# Vérifier que tous les fichiers existent
test_df["exists"] = test_df["image_path"].map(
    lambda path: Path(path).is_file()
)

print("Images attendues :", len(test_df))
print("Images trouvées  :", test_df["exists"].sum())
print("Images manquantes :", (~test_df["exists"]).sum())

if not test_df["exists"].all():
    display(
        test_df.loc[
            ~test_df["exists"],
            ["class", "filename", "image_path"]
        ].head(10)
    )

assert len(test_df) == 4233, "Nombre d'images inattendu"
assert test_df["exists"].all(), "Certaines images sont introuvables"

print("\n✓ Jeu de test prêt : 4 233 images accessibles.")

Images attendues : 4233
Images trouvées  : 4233
Images manquantes : 0

✓ Jeu de test prêt : 4 233 images accessibles.


In [9]:
from pathlib import Path
import shutil
import time
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/Apprentissage/DS/projet")

DRIVE_DATASET_ROOT = PROJECT_DIR / "COVID-19_Radiography_Dataset"
DATASET_ROOT = Path("/content/COVID-19_Radiography_Dataset")

# 1. Copie locale, une seule fois par session Colab
assert DRIVE_DATASET_ROOT.is_dir(), "Dataset introuvable sur Drive"

if not DATASET_ROOT.is_dir():
    start = time.perf_counter()
    shutil.copytree(DRIVE_DATASET_ROOT, DATASET_ROOT)
    print(f"Copie terminée en {(time.perf_counter() - start)/60:.1f} min")
else:
    print("Dataset local déjà présent :", DATASET_ROOT)

# 2. Chargement du split de test
test_df = pd.read_csv(PROJECT_DIR / "test.csv")

# 3. Conversion des chemins Windows vers Colab
def local_filepath(original):
    normalized = str(original).replace("\\", "/")
    marker = "COVID-19_Radiography_Dataset/"

    if marker not in normalized:
        raise ValueError(f"Chemin inattendu : {original}")

    relative_path = normalized.split(marker, 1)[1]
    return str(DATASET_ROOT / relative_path)

test_df["image_path"] = test_df["filepath"].apply(local_filepath)

# 4. Vérification
missing = test_df[
    ~test_df["image_path"].map(lambda p: Path(p).is_file())
]

print("Images attendues :", len(test_df))
print("Images manquantes :", len(missing))

assert len(test_df) == 4233
assert missing.empty, "Certaines images sont introuvables"

print("✓ Les 4 233 images du test sont accessibles localement.")

Copie terminée en 14.1 min
Images attendues : 4233
Images manquantes : 0
✓ Les 4 233 images du test sont accessibles localement.


In [10]:
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input

PROJECT_DIR = Path("/content/drive/MyDrive/Apprentissage/DS/projet")
DATASET_ROOT = Path("/content/COVID-19_Radiography_Dataset")

OUTPUT_DIR = PROJECT_DIR / "benchmark_final_test_E4_M4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Jeu de test commun
test_df = pd.read_csv(PROJECT_DIR / "test.csv")

def local_filepath(original):
    normalized = str(original).replace("\\", "/")
    marker = "COVID-19_Radiography_Dataset/"

    if marker not in normalized:
        raise ValueError(f"Chemin inattendu : {original}")

    return str(DATASET_ROOT / normalized.split(marker, 1)[1])

test_df["filepath"] = test_df["filepath"].map(local_filepath)
test_df["target"] = test_df["target"].astype(str)
test_df["class"] = test_df["class"].astype(str)

# Contrôles communs
assert len(test_df) == 4233
assert test_df["filepath"].map(lambda p: Path(p).is_file()).all()
assert (test_df["target"].eq("1") == test_df["class"].eq("COVID")).all()

# Prétraitement commun aux deux VGG16
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

print("✓ Jeu de test :", len(test_df), "images")
print("✓ Chemins locaux vérifiés")
print("✓ Prétraitement VGG16 prêt")
print("GPU :", tf.config.list_physical_devices("GPU"))

✓ Jeu de test : 4233 images
✓ Chemins locaux vérifiés
✓ Prétraitement VGG16 prêt
GPU : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [11]:
# ============================================
# E4 — VGG16 binaire, seuil COVID = 0.35
# ============================================

E4_CHECKPOINT = (
    PROJECT_DIR / "E4_finetuning"
    / "E4_FT_B5_LR1e-5_best.keras"
)

assert E4_CHECKPOINT.is_file()

# Générateur E4 : classes 0 / 1
gen_e4 = datagen.flow_from_dataframe(
    test_df,
    x_col="filepath",
    y_col="target",
    classes=["0", "1"],
    target_size=(224, 224),
    color_mode="rgb",
    class_mode="binary",
    batch_size=32,
    shuffle=False
)

assert gen_e4.class_indices == {"0": 0, "1": 1}
assert gen_e4.samples == 4233
assert np.array_equal(
    gen_e4.classes,
    test_df["target"].astype(int).to_numpy()
)

# Chargement du meilleur checkpoint
model_e4 = load_model(E4_CHECKPOINT, compile=False)
assert model_e4.output_shape[-1] == 1

# Prédictions
gen_e4.reset()
prob_e4 = np.asarray(
    model_e4.predict(gen_e4, verbose=1)
).reshape(-1)

assert prob_e4.shape == (4233,)
assert np.isfinite(prob_e4).all()
assert np.all((prob_e4 >= 0) & (prob_e4 <= 1))

# Règle de décision figée sur validation
pred_e4 = (prob_e4 >= 0.35).astype(int)

# Sauvegarde E4 uniquement
results_e4 = test_df[
    ["class", "filename", "target", "filepath"]
].copy()

results_e4["target"] = results_e4["target"].astype(int)
results_e4["E4_prob_COVID"] = prob_e4
results_e4["E4_pred_COVID_seuil_035"] = pred_e4

output_e4 = OUTPUT_DIR / "predictions_test_E4.csv"
results_e4.to_csv(output_e4, index=False)

print("\n✓ E4 terminé :", len(results_e4), "prédictions")
print("✓ CSV :", output_e4)

display(results_e4.head())

del model_e4
tf.keras.backend.clear_session()

Found 4233 validated image filenames belonging to 2 classes.
133/133 ━━━━━━━━━━━━━━━━━━━━ 30s 196ms/step

✓ E4 terminé : 4233 prédictions
✓ CSV : /content/drive/MyDrive/Apprentissage/DS/projet/benchmark_final_test_E4_M4/predictions_test_E4.csv


,class,filename,target,filepath,E4_prob_COVID,E4_pred_COVID_seuil_035
0,Lung_Opacity,Lung_Opacity-121.png,0,/content/COVID-19_Radiography_Dataset/Lung_Opa...,1.055867e-09,0
1,Normal,Normal-828.png,0,/content/COVID-19_Radiography_Dataset/Normal/i...,8.655540e-01,1
2,Lung_Opacity,Lung_Opacity-4959.png,0,/content/COVID-19_Radiography_Dataset/Lung_Opa...,1.613942e-06,0
3,Lung_Opacity,Lung_Opacity-405.png,0,/content/COVID-19_Radiography_Dataset/Lung_Opa...,2.679703e-04,0
4,Normal,Normal-3816.png,0,/content/COVID-19_Radiography_Dataset/Normal/i...,2.145187e-03,0


In [12]:
# ============================================
# M4 — VGG16 multiclasses, décision argmax
# ============================================

M4_CHECKPOINT = (
    PROJECT_DIR / "M4_finetuning"
    / "M4_FT2_B5_LR1e-6_best.keras"
)

assert M4_CHECKPOINT.is_file()

# Ordre des classes identique à l'entraînement M4
classes_m4 = sorted(
    pd.read_csv(PROJECT_DIR / "train.csv")["class"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

assert len(classes_m4) == 4
assert "COVID" in classes_m4

# Générateur M4 : quatre classes
gen_m4 = datagen.flow_from_dataframe(
    test_df,
    x_col="filepath",
    y_col="class",
    classes=classes_m4,
    target_size=(224, 224),
    color_mode="rgb",
    class_mode="categorical",
    batch_size=32,
    shuffle=False
)

assert gen_m4.samples == 4233
assert np.array_equal(
    gen_m4.classes,
    test_df["class"].map(gen_m4.class_indices).to_numpy()
)

print("Mapping M4 :", gen_m4.class_indices)

# Chargement du meilleur checkpoint phase 2
model_m4 = load_model(M4_CHECKPOINT, compile=False)
assert model_m4.output_shape[-1] == 4

# Prédictions
gen_m4.reset()
prob_m4 = np.asarray(
    model_m4.predict(gen_m4, verbose=1)
)

assert prob_m4.shape == (4233, 4)
assert np.isfinite(prob_m4).all()

# Règle de décision figée sur validation
pred_m4_idx = np.argmax(prob_m4, axis=1)
pred_m4_class = np.array(classes_m4)[pred_m4_idx]
pred_m4_covid = (pred_m4_class == "COVID").astype(int)

# Sauvegarde M4 uniquement
results_m4 = test_df[
    ["class", "filename", "target", "filepath"]
].copy()

results_m4["target"] = results_m4["target"].astype(int)

for i, cls in enumerate(classes_m4):
    results_m4[f"M4_prob_{cls}"] = prob_m4[:, i]

results_m4["M4_pred_classe_argmax"] = pred_m4_class
results_m4["M4_pred_COVID_argmax"] = pred_m4_covid

output_m4 = OUTPUT_DIR / "predictions_test_M4.csv"
results_m4.to_csv(output_m4, index=False)

print("\n✓ M4 terminé :", len(results_m4), "prédictions")
print("✓ CSV :", output_m4)

display(results_m4.head())

del model_m4
tf.keras.backend.clear_session()

Found 4233 validated image filenames belonging to 4 classes.
Mapping M4 : {'COVID': 0, 'Lung_Opacity': 1, 'Normal': 2, 'Viral Pneumonia': 3}
133/133 ━━━━━━━━━━━━━━━━━━━━ 27s 197ms/step

✓ M4 terminé : 4233 prédictions
✓ CSV : /content/drive/MyDrive/Apprentissage/DS/projet/benchmark_final_test_E4_M4/predictions_test_M4.csv


,class,filename,target,filepath,M4_prob_COVID,M4_prob_Lung_Opacity,M4_prob_Normal,M4_prob_Viral Pneumonia,M4_pred_classe_argmax,M4_pred_COVID_argmax
0,Lung_Opacity,Lung_Opacity-121.png,0,/content/COVID-19_Radiography_Dataset/Lung_Opa...,3.941562e-08,0.999980,1.980137e-05,1.884731e-14,Lung_Opacity,0
1,Normal,Normal-828.png,0,/content/COVID-19_Radiography_Dataset/Normal/i...,9.922852e-01,0.000038,7.676672e-03,7.921577e-08,COVID,1
2,Lung_Opacity,Lung_Opacity-4959.png,0,/content/COVID-19_Radiography_Dataset/Lung_Opa...,7.107040e-08,1.000000,4.158809e-10,8.045615e-18,Lung_Opacity,0
3,Lung_Opacity,Lung_Opacity-405.png,0,/content/COVID-19_Radiography_Dataset/Lung_Opa...,1.921731e-03,0.990453,7.625228e-03,9.006538e-09,Lung_Opacity,0
4,Normal,Normal-3816.png,0,/content/COVID-19_Radiography_Dataset/Normal/i...,4.318514e-05,0.000286,9.996704e-01,2.608418e-12,Normal,0


In [14]:
# ============================================================
# BENCHMARK FINAL SUR TEST — E4 vs M4
# Évaluation uniquement : aucune prédiction ni optimisation
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

PROJECT_DIR = Path("/content/drive/MyDrive/Apprentissage/DS/projet")
OUTPUT_DIR = PROJECT_DIR / "benchmark_final_test_E4_M4"

# ------------------------------------------------------------
# 1. Chargement des prédictions sauvegardées
# ------------------------------------------------------------

e4 = pd.read_csv(OUTPUT_DIR / "predictions_test_E4.csv")
m4 = pd.read_csv(OUTPUT_DIR / "predictions_test_M4.csv")

assert len(e4) == len(m4) == 4233

# Vérifier que les prédictions concernent les mêmes images,
# dans le même ordre.
assert e4["filepath"].equals(m4["filepath"])
assert e4["class"].equals(m4["class"])
assert e4["target"].equals(m4["target"])

print("✓ 4 233 images de test")
print("✓ Alignement E4 / M4 vérifié")

# Vérité terrain
y_true = e4["target"].astype(int).to_numpy()

# Prédictions COVID / non-COVID
y_pred_e4 = e4["E4_pred_COVID_seuil_035"].astype(int).to_numpy()
y_pred_m4 = m4["M4_pred_COVID_argmax"].astype(int).to_numpy()

# Classes multiclasses M4
classes_m4 = sorted(
    pd.read_csv(PROJECT_DIR / "train.csv")["class"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

# ------------------------------------------------------------
# 2. Fonction de calcul des métriques COVID
# ------------------------------------------------------------

def metrics_covid(y_true, y_pred, modele):

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[1],
        average=None,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    return {
        "Modele": modele,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_COVID": precision[0],
        "Recall_COVID": recall[0],
        "F1_COVID": f1[0],
        "TP": int(tp),
        "FN": int(fn),
        "FP": int(fp),
        "TN": int(tn)
    }

# ------------------------------------------------------------
# 3. Tableau comparatif E4 vs M4
# ------------------------------------------------------------

comparison = pd.DataFrame([
    metrics_covid(y_true, y_pred_e4, "E4 — seuil 0.35"),
    metrics_covid(y_true, y_pred_m4, "M4 — argmax")
])

print("\n=== BENCHMARK FINAL — COVID VS NON-COVID ===")

display(
    comparison.style.format({
        "Accuracy": "{:.2%}",
        "Precision_COVID": "{:.2%}",
        "Recall_COVID": "{:.2%}",
        "F1_COVID": "{:.2%}"
    })
)

comparison.to_csv(
    OUTPUT_DIR / "comparaison_finale_test_E4_M4.csv",
    index=False
)

# ------------------------------------------------------------
# 4. Rapport binaire E4
# ------------------------------------------------------------

print("\n=== RAPPORT E4 — COVID VS NON-COVID ===")

report_e4 = classification_report(
    y_true,
    y_pred_e4,
    labels=[0, 1],
    target_names=["Non-COVID", "COVID"],
    digits=4,
    zero_division=0
)

print(report_e4)

report_e4_dict = classification_report(
    y_true,
    y_pred_e4,
    labels=[0, 1],
    target_names=["Non-COVID", "COVID"],
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report_e4_dict).transpose().to_csv(
    OUTPUT_DIR / "rapport_binaire_test_E4.csv"
)

# ------------------------------------------------------------
# 5. Rapport binaire M4
# ------------------------------------------------------------

print("\n=== RAPPORT M4 — COVID VS NON-COVID ===")

report_m4_binary = classification_report(
    y_true,
    y_pred_m4,
    labels=[0, 1],
    target_names=["Non-COVID", "COVID"],
    digits=4,
    zero_division=0
)

print(report_m4_binary)

report_m4_binary_dict = classification_report(
    y_true,
    y_pred_m4,
    labels=[0, 1],
    target_names=["Non-COVID", "COVID"],
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report_m4_binary_dict).transpose().to_csv(
    OUTPUT_DIR / "rapport_binaire_test_M4.csv"
)

# ------------------------------------------------------------
# 6. Rapport multiclasses M4
# ------------------------------------------------------------

print("\n=== RAPPORT M4 — QUATRE CLASSES ===")

y_true_m4 = m4["class"].astype(str).to_numpy()
y_pred_m4_class = m4["M4_pred_classe_argmax"].astype(str).to_numpy()

report_m4_multi = classification_report(
    y_true_m4,
    y_pred_m4_class,
    labels=classes_m4,
    digits=4,
    zero_division=0
)

print(report_m4_multi)

report_m4_multi_dict = classification_report(
    y_true_m4,
    y_pred_m4_class,
    labels=classes_m4,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report_m4_multi_dict).transpose().to_csv(
    OUTPUT_DIR / "rapport_multiclasses_test_M4.csv"
)

# ------------------------------------------------------------
# 7. Matrices de confusion
# ------------------------------------------------------------

cm_e4 = confusion_matrix(
    y_true, y_pred_e4, labels=[0, 1]
)

cm_m4_binary = confusion_matrix(
    y_true, y_pred_m4, labels=[0, 1]
)

cm_m4_multi = confusion_matrix(
    y_true_m4,
    y_pred_m4_class,
    labels=classes_m4
)

print("\n=== MATRICE E4 — BINAIRE ===")
print(cm_e4)

print("\n=== MATRICE M4 — BINAIRE ===")
print(cm_m4_binary)

print("\n=== MATRICE M4 — MULTICLASSES ===")
print(cm_m4_multi)

pd.DataFrame(
    cm_e4,
    index=["Reel_NonCOVID", "Reel_COVID"],
    columns=["Predit_NonCOVID", "Predit_COVID"]
).to_csv(OUTPUT_DIR / "matrice_confusion_test_E4.csv")

pd.DataFrame(
    cm_m4_binary,
    index=["Reel_NonCOVID", "Reel_COVID"],
    columns=["Predit_NonCOVID", "Predit_COVID"]
).to_csv(OUTPUT_DIR / "matrice_confusion_binaire_test_M4.csv")

pd.DataFrame(
    cm_m4_multi,
    index=classes_m4,
    columns=classes_m4
).to_csv(OUTPUT_DIR / "matrice_confusion_multiclasses_test_M4.csv")

# ------------------------------------------------------------
# 8. Récapitulatif des fichiers enregistrés
# ------------------------------------------------------------

print("\n=== FICHIERS ENREGISTRÉS ===")

output_files = [
    "comparaison_finale_test_E4_M4.csv",
    "rapport_binaire_test_E4.csv",
    "rapport_binaire_test_M4.csv",
    "rapport_multiclasses_test_M4.csv",
    "matrice_confusion_test_E4.csv",
    "matrice_confusion_binaire_test_M4.csv",
    "matrice_confusion_multiclasses_test_M4.csv"
]

for filename in output_files:
    path = OUTPUT_DIR / filename
    assert path.is_file()
    print("✓", filename)

print("\n✓ Benchmark final E4/M4 terminé.")
print("Dossier :", OUTPUT_DIR)

✓ 4 233 images de test
✓ Alignement E4 / M4 vérifié

=== BENCHMARK FINAL — COVID VS NON-COVID ===


,Modele,Accuracy,Precision_COVID,Recall_COVID,F1_COVID,TP,FN,FP,TN
0,E4 — seuil 0.35,98.82%,95.66%,97.51%,96.58%,705,18,32,3478
1,M4 — argmax,98.80%,96.67%,96.27%,96.47%,696,27,24,3486



=== RAPPORT E4 — COVID VS NON-COVID ===
              precision    recall  f1-score   support

   Non-COVID     0.9949    0.9909    0.9929      3510
       COVID     0.9566    0.9751    0.9658       723

    accuracy                         0.9882      4233
   macro avg     0.9757    0.9830    0.9793      4233
weighted avg     0.9883    0.9882    0.9882      4233


=== RAPPORT M4 — COVID VS NON-COVID ===
              precision    recall  f1-score   support

   Non-COVID     0.9923    0.9932    0.9927      3510
       COVID     0.9667    0.9627    0.9647       723

    accuracy                         0.9880      4233
   macro avg     0.9795    0.9779    0.9787      4233
weighted avg     0.9879    0.9880    0.9879      4233


=== RAPPORT M4 — QUATRE CLASSES ===
                 precision    recall  f1-score   support

          COVID     0.9667    0.9627    0.9647       723
   Lung_Opacity     0.9101    0.8965    0.9032      1140
         Normal     0.9389    0.9483    0.9436      209